# 



Seleção de Atributos com Algoritmo Genético e Random Forest
Este notebook executa seleção de atributos utilizando algoritmo genético com avaliação por Random Forest.

In [1]:
import pandas as pd
import numpy as np
import random
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix, accuracy_score, recall_score
from scipy.stats import entropy
from deap import base, creator, tools, algorithms


In [2]:
def eval_individual(individual):
    if sum(individual) == 0:
        return 0.0,

    selected_cols = [i for i, bit in enumerate(individual) if bit == 1]
    model = RandomForestClassifier(class_weight='balanced', n_estimators=100, n_jobs=-1, random_state=42)
    model.fit(X_train.iloc[:, selected_cols], y_train)
    preds = model.predict(X_test.iloc[:, selected_cols])

    return recall_score(y_test, preds, pos_label=1.0),

In [3]:
df = pd.read_csv("database-5.csv")
classe = "Q092"

In [4]:
X = df.drop(columns=[classe])
y = df[classe]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [5]:
N_FEATURES = X.shape[1]

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=N_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)


In [6]:
toolbox.register("evaluate", eval_individual)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)


In [7]:
pop = toolbox.population(n=40)
result_pop, logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=100, verbose=True)


gen	nevals
0  	40    
1  	23    
2  	21    
3  	22    
4  	25    
5  	29    
6  	31    
7  	23    
8  	27    
9  	26    
10 	35    
11 	27    
12 	21    
13 	22    
14 	27    
15 	26    
16 	24    
17 	31    
18 	24    
19 	28    
20 	27    
21 	20    
22 	22    
23 	30    
24 	24    
25 	23    
26 	23    
27 	27    
28 	24    
29 	29    
30 	21    
31 	25    
32 	27    
33 	21    
34 	30    
35 	30    
36 	23    
37 	27    
38 	18    
39 	27    
40 	21    
41 	14    
42 	20    
43 	30    
44 	25    
45 	22    
46 	25    
47 	26    
48 	23    
49 	25    
50 	24    
51 	18    
52 	19    
53 	16    
54 	22    
55 	28    
56 	30    
57 	21    
58 	31    
59 	29    
60 	26    
61 	21    
62 	24    
63 	29    
64 	24    
65 	20    
66 	21    
67 	21    
68 	20    
69 	26    
70 	27    
71 	23    
72 	26    
73 	21    
74 	20    
75 	29    
76 	26    
77 	23    
78 	21    
79 	27    
80 	22    
81 	21    
82 	24    
83 	23    
84 	27    
85 	29    
86 	22    
87 	22    
88 	21    
89 	24    

In [8]:
# Melhor indivíduo encontrado
best_ind = tools.selBest(result_pop, k=1)[0]
print("Melhor conjunto de atributos (máscara binária):", best_ind)

selected_columns = [X.columns[i] for i, bit in enumerate(best_ind) if bit == 1]
print("Atributos selecionados:")
for i, nome in enumerate(selected_columns, 1):
    print(f"{i:>2}. {nome}")

# Modelo final
final_model = RandomForestClassifier(class_weight='balanced', n_estimators=100, n_jobs=-1, random_state=42)
final_model.fit(X_train[selected_columns], y_train)
final_preds = final_model.predict(X_test[selected_columns])
metricas = extrair_metricas(y_test, final_preds, nome="Random Forest")
print("-" * 32)
for chave, valor in metricas.items():
    print(f"{chave:<15}: {valor:>15.4f}" if isinstance(valor, float) else f"{chave:<15}: {valor}")


Melhor conjunto de atributos (máscara binária): [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0]
Atributos selecionados:
 1. C006
 2. J05301
 3. P03302
 4. P03303
 5. Tempo_Minimo_Exercicio
 6. Violencia_Fisica


NameError: name 'extrair_metricas' is not defined

<div align="center">

### ✅ Métricas do Modelo (`Random Forest`)

| Métrica     | Valor   |
|-------------|---------|
| **Accuracy** | 0.6675  |
| **F1 Macro** | 0.5555  |

#### Classe 1.0 (Diagnóstico Positivo)

| Métrica   | Valor   |
|-----------|---------|
| Precision | 0.2135  |
| Recall    | 0.7486  |
| F1-Score  | 0.3323  |

#### Classe 2.0 (Diagnóstico Negativo)

| Métrica   | Valor   |
|-----------|---------|
| Precision | 0.9546  |
| Recall    | 0.6574  |
| F1-Score  | 0.7786  |

</div>

### Atributos selecionados pelo Algoritmo Genético

1. A001
2. C006
3. J05301
4. Q03001
5. Violencia_Fisica
6. Deslocamento_Trabalho_Dias